# Importing libraries and set style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

import pickle

from functions.functions import *

Style setting

In [ ]:
plt.style.use('seaborn-v0_8')
# plt.style.use('seaborn-v0_8-colorblind')
# sns.set_theme(style='whitegrid')

In [ ]:
# Figure
# plt.rcParams['figure.figsize'] = (8, 6)
# plt.rcParams['figure.dpi'] = 300

# Title
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# Axes
plt.rcParams['axes.labelsize'] = 12

Font setting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Nimbus Sans']

# Import data

In [ ]:
df = pd.read_parquet('../raw_data/parquets/1-escribe.parquet')

# with open('../raw_data/pickles/.pickle', 'rb') as file:
# 	df = pickle.load(file)

# Filter the dataset by specifying a start and/or end date

In [ ]:
datecol = 'acq_time'
start = '2023-01-17 09:39:00'
end = '2023-01-23 09:39:00'

In [ ]:
# df = df_filter_by_date(df, datecol,
#     start=start,
#     end=end
# )

# Overview on all the data

In [ ]:
# df

In [ ]:
df.info()

# Functions

In [ ]:
def bar_plot(ser,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, kind='unknown', *, label_limit=100):

    if kind == 'unknown':
        raise Exception(f'Kind {kind} not valid')

    # Create figure
    fig, ax = plt.subplots()
    bars = ax.bar(ser.index.astype(str), ser.values)
    ax.bar_label(bars,
        labels=np.where(ser.values < label_limit, ser.values, ''),
        fontsize=9
    )

    if kind == 'date':
        pass
        # fig.autofmt_xdate()
    elif kind == 'time':
        pass
        # ax.set_xticks(ser.index)

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

    plt.tight_layout()
    # plt.savefig(f'{output_dir}/{progr}-{output_name}.{output_format}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def line_plot(ser,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format):


    original_len = len(df[colname])
    data = df[colname].dropna()
    na_count = original_len - len(data)

    per_year = data.dt.year.value_counts().sort_index()

    # Create figure
    fig, ax = plt.subplots()
    ax.plot(per_year.index, per_year.values)

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)


    n_total = original_len
    n_valid = len(data)

    ax.text(
        0.98, 0.95,
        f'n = {n_valid} (of {n_total})\n{round((n_valid/n_total)*100)} %',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=11
    )

    plt.tight_layout()
    # plt.savefig(f'{output_dir}/{progr}-{output_name}.{output_format}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def histbox_plot(ser,
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto'):

    original_len = len(df[colname])
    data = df[colname].dropna()
    na_count = original_len - len(data)

    if na_count > 0:
        print(f'Dropped {na_count} NA values ({na_count / original_len:.1%} of total)')

    # Create figure with 2 rows (hist + boxplot)
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=figsize,
        sharex=True,
        gridspec_kw={'height_ratios': [4, 1]}
    )

    # Histogram
    ax1.hist(data, bins=bins)
    ax1.set_ylabel(y_label)
    ax1.set_title(title)

    # Info about total and NAs
    n_total = original_len
    n_valid = len(data)

    ax1.text(
        0.98, 0.95,
        f'n = {n_valid} (of {n_total})\n{round((n_valid/n_total)*100)} %',
        transform=ax1.transAxes,
        ha='right', va='top',
        fontsize=11
    )

    # Boxplot
    ax2.boxplot(data, vert=False, widths=0.6)
    ax2.set_xlabel(x_label)
    ax2.set_yticks([])  # optional: hide y-axis ticks

    # Statistical info
    stats_info = {}
    # Outliers
    fliers = ax2.boxplot(data, vert=False, widths=0.6)['fliers'][0].get_xdata()
    # convert to Python int
    fliers = [int(v) for v in fliers]
    outlier_counts = dict(Counter(fliers))
    stats_info['outlier_counts'] = outlier_counts
    # Mean
    mean = round(data.mean(), 3)
    stats_info['mean'] = mean
    # stddev
    stddev = round(data.std(), 3)
    stats_info['stddev'] = stddev
    # Print
    print(f'Outliers: {outlier_counts}')
    print(f'Mean: {mean}')
    print(f'Std dev: {stddev}')

    # Savefig
    plt.tight_layout()
    # plt.savefig(f'{output_dir}/{progr}-{output_name}.{output_format}', dpi=300, bbox_inches='tight')
    plt.show()

# Plots

**Common values**

In [ ]:
output_dir = '../plots/1-escribe'
figsize = (8, 6)
progr = 0
output_format = 'pdf'

## Line plots

#### `patient_date_of_birth`

In [ ]:
colname = 'patient_date_of_birth'
title = 'Births per year'
x_label = 'Year'
y_label = 'Number of patients'
output_name = 'birth_years'
progr += 1

In [ ]:
line_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format)

#### `acq_time`

In [ ]:
colname = 'acq_time'
title = 'Acquisitions per year'
x_label = 'Year'
y_label = 'Number of patients'
output_name = 'acq_time'
progr += 1

In [ ]:
line_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format)

## Hist-bar plots

In [ ]:
figsize = (8, 6)
y_label = 'Count'

#### `patient_age`

In [ ]:
colname = 'patient_age'
title = 'Age Distribution'
x_label = 'Age'
# y_label = ''
output_name = 'age_distribution'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

### ECGS

#### Temporal (wave timing) features - measured in ms

In [ ]:
x_label = 'milliseconds'

##### `p_onset`

In [ ]:
colname = 'p_onset'
title = 'p_onset'
# x_label = ''
# y_label = ''
output_name = 'p_onset'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `p_offset`

In [ ]:
colname = 'p_offset'
title = 'p_offset'
# x_label = ''
# y_label = ''
output_name = 'p_offset'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `qrs_onset`

In [ ]:
colname = 'qrs_onset'
title = 'qrs_onset'
# x_label = ''
# y_label = ''
output_name = 'qrs_onset'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `qrs_offset`

In [ ]:
colname = 'qrs_offset'
title = 'qrs_offset'
# x_label = ''
# y_label = ''
output_name = 'qrs_offset'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `t_offset`

In [ ]:
colname = 't_offset'
title = 't_offset'
# x_label = ''
# y_label = ''
output_name = 't_offset'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

#### Interval/duration features - measured in ms

In [ ]:
x_label = 'milliseconds'

##### `pr_interval`

In [ ]:
colname = 'pr_interval'
title = 'pr_interval'
# x_label = ''
# y_label = ''
output_name = 'pr_interval'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `qrs_duration`

In [ ]:
colname = 'qrs_duration'
title = 'qrs_duration'
# x_label = ''
# y_label = ''
output_name = 'qrs_duration'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `qt_interval`

In [ ]:
colname = 'qt_interval'
title = 'qt_interval'
# x_label = ''
# y_label = ''
output_name = 'qt_interval'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `qtc_interval`

In [ ]:
colname = 'qtc_interval'
title = 'qtc_interval'
# x_label = ''
# y_label = ''
output_name = 'qtc_interval'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

##### `rr_interval`

In [ ]:
colname = 'rr_interval'
title = 'rr_interval'
# x_label = ''
# y_label = ''
output_name = 'rr_interval'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

#### Axis (vector direction) features - measured in deg

In [ ]:
x_label = 'degrees'

#### `p_axis`

In [ ]:
colname = 'p_axis'
title = 'p_axis'
# x_label = ''
# y_label = ''
output_name = 'p_axis'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

#### `qrs_axis`

In [ ]:
colname = 'qrs_axis'
title = 'qrs_axis'
# x_label = ''
# y_label = ''
output_name = 'qrs_axis'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

#### `t_axis`

In [ ]:
colname = 't_axis'
title = 't_axis'
# x_label = ''
# y_label = ''
output_name = 't_axis'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')

#### Heart rate feature - measured in bpm

In [ ]:
x_label = 'bpm'

#### `v_rate`

In [ ]:
colname = 'v_rate'
title = 'v_rate'
# x_label = ''
# y_label = ''
output_name = 'v_rate'
progr += 1

In [ ]:
histbox_plot(df[colname],
    title, x_label, y_label,
    output_dir, progr, output_name, output_format,
    figsize, bins='auto')